In [1]:
import os
import glob

import json
import rasterio
from rasterio.mask import mask
from rasterio.windows import Window
import numpy as np 
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon,box

In [2]:
FILEPATH = r'C:\Users\b1114674\Desktop\thesis\AI4EO_data\international-charter-eo4ai-etq-challenge\Training'

EXCELPATH = r'C:\Users\b1114674\Desktop\thesis\AI4EO_data\international-charter-eo4ai-etq-challenge\Training\Phase_1_Dataset_Details.xlsx'

DESTINATION_PRE_PATH = r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\PRE'
DESTINATION_POST_PATH = r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\POST'

In [29]:
class GEO_PreProssessing:

    def __init__(self):
          pass
        

    def open(self,path,no_channels=[1,2,3],transpose=True):
        '''
        Read the image 

        Args:
        path(str): filepath to the image
        no_channels(list(int)): how many channels do we need
        transpose(bool): True to change the shape of image to (H,W,C)

        Return: Image in numpy array format as (H,W,C) and Source format
        '''
        src = rasterio.open(path)
        img = src.read(no_channels)
        if transpose:
              img.transpose(1,2,0)
              return img,src
        else:
              return img,src

    def findOverlap(self,in_pre,in_post,out_pre,out_post,no_channels=[1,2,3]):
        
        '''
        Find the overlapping area an save both raster within intersection extent

        Args:
        in_Pre(str) : Absolute filepath to the pre image file
        in_post(str) : Absolute filepath to the post image file
        out_pre(str) : Absolute filepath to the Save cropped pre image file
        out_post(str) : Absolute filepath to the Save cropped post image file
        no_channels(list(int)): how many channels do we need
        
        '''

        _,pre_src = self.open(in_pre,no_channels)
        pre_bounds = box(*pre_src.bounds)
        crs = pre_src.crs
        
        _,post_src = self.open(in_post,no_channels)
        post_bounds = box(*post_src.bounds)

        common_area = pre_bounds.intersection(post_bounds)
    

        assert not common_area.is_empty, f'No Overlapping areas in the given images {os.path.basename(in_pre)} and {os.path.basename(in_post)}'

        # Converts Shapely geometry to GeoJSON format
        common_geojson = [json.loads(gpd.GeoSeries([common_area], crs=crs).to_json())['features'][0]['geometry']]

        pre_clip, pre_transform = mask(pre_src, common_geojson, crop=True)
        pre_meta = pre_src.meta.copy()
        pre_meta.update({
            "height": pre_clip.shape[1],
            "width": pre_clip.shape[2],
            "count": pre_clip.shape[0],
            "transform": pre_transform
            })
    

        post_clip, post_transform = mask(post_src, common_geojson, crop=True)
        post_meta = post_src.meta.copy()
        post_meta.update({
            "height": post_clip.shape[1],
            "width": post_clip.shape[2],
            "count": post_clip.shape[0],
            "transform": post_transform
            })
        
        try:
            with rasterio.open(out_pre,'w',**pre_meta) as pre_dest:
                                pre_dest.write(pre_clip)

            with rasterio.open(out_post,'w',**post_meta) as post_dest:
                                post_dest.write(post_clip)

        except ValueError:
              print('Invalid filepath!')


    def makeChips(self,in_pre,in_post,out_pre,out_post,no_channels=[1,2,3],chip_size=224,stride=224):    

        '''

        Args:
        in_Pre(str) : Absolute filepath to the pre image file
        in_post(str) : Absolute filepath to the post image file
        out_pre(str) : Absolute filepath to the Save cropped pre image file
        out_post(str) : Absolute filepath to the Save cropped post image file
        no_channels(list(int)): how many channels do we need
        chip_size(int): The size of the image height and width
        stride(int): step size or overlap between the chips
        
        '''
        with rasterio.open(in_pre) as pre_src, rasterio.open(in_post) as post_src:
            counter = 0

            if pre_src.height == post_src.height and pre_src.width == post_src.width:

                for row in range(0,pre_src.height,stride):
                        for col in range(0,pre_src.width,stride):
                            window = Window(col,row,chip_size,chip_size)

                            pre_img = pre_src.read(no_channels,window=window)
                            post_img = post_src.read(no_channels,window=window)

                            if row + chip_size > pre_src.height or col + chip_size > pre_src.width:
                                continue

                            pre_meta = pre_src.meta.copy()
                            pre_meta.update({
                                "height":pre_img.shape[1],
                                "width":pre_img.shape[2],
                                "count":pre_img.shape[0],
                                "transform":rasterio.windows.transform(window,pre_src.transform)  
                            })

                            post_meta = post_src.meta.copy()
                            post_meta.update({
                                "height":post_img.shape[1],
                                "width":post_img.shape[2],
                                "count":post_img.shape[0],
                                "transform":rasterio.windows.transform(window,post_src.transform)  
                            })
                            
                            try:

                                if os.path.exists(out_pre) and os.path.exists(out_post):

                                    with rasterio.open(os.path.join(out_pre,f"pre{counter}.tif"),"w",**pre_meta) as pre_dest:
                                        pre_dest.write(pre_img)

                                    with rasterio.open(os.path.join(out_post,f"post{counter}.tif"),"w",**post_meta) as post_dest:
                                        post_dest.write(post_img)

                                    counter+=1

                            except ValueError:
                                print("Invalid path!")
            else:
                print('[OVERLAP ERROR]----- MISMATCH IN IMAGE SIZES')

                crs = pre_src.crs
                pre_meta = pre_src.meta.copy()
                post_meta = post_src.meta.copy()
                pre_bound = box(*pre_src.bounds)
                post_bound = box(*post_src.bounds)

                common_area = pre_bound.intersection(post_bound)

                common_geojson = [json.loads(gpd.GeoSeries([common_area], crs=crs).to_json())['features'][0]['geometry']]

                print(f'Before PRE crop {pre_src.height,pre_src.width,pre_src.count}')

                print(f'Before POST crop {post_src.height,post_src.width,post_src.count}')


                pre_clip,pre_transform = mask(pre_src,common_geojson,crop=True)
                post_clip,post_transform = mask(post_src,common_geojson,crop=True)

                print('IMAGE CROPPED TO A COMMON SIZE')

                assert pre_clip.shape[1] == post_clip.shape[1] and pre_clip.shape[2] == post_clip.shape[2], f"ISSUE WITH THE IMAGE SIZING, TRY TO CUT THE IMAGE INTO A COMMAN SIZE {pre_clip.shape} and {post_clip.shape}"

                for i in range(0,pre_clip.shape[1],stride):
                        for j in range(0,pre_clip.shape[2],stride):
                            pre_chip = pre_clip[:len(no_channels),i:i+chip_size,j:j+chip_size]
                            post_chip= post_clip[:len(no_channels),i:i+chip_size,j:j+chip_size]

                            pre_meta.update({
                                    "height":pre_chip.shape[1],
                                "width":pre_chip.shape[2],
                                "count":pre_chip.shape[0],
                                "transform": pre_transform
                                    
                            })

                            post_meta.update({
                                    "height":post_chip.shape[1],
                                "width":post_chip.shape[2],
                                "count":post_chip.shape[0],
                                "transform": post_transform
                                    
                            })

                            try:

                                if os.path.exists(out_pre) and os.path.exists(out_post):

                                    with rasterio.open(os.path.join(out_pre,f"pre{counter}.tif"),"w",**pre_meta) as pre_dest:
                                        pre_dest.write(pre_chip)

                                    with rasterio.open(os.path.join(out_post,f"post{counter}.tif"),"w",**post_meta) as post_dest:
                                        post_dest.write(post_chip)

                                    counter+=1

                            except ValueError:
                                print("Invalid path!")
                                
                                
    def Mean_Std(self,IMAGE_FOLDER_PATH,no_channels):
        '''
        Computing the Mean and Std for the whole dataset

        Args:
            IMAGE_FOLDER_PATH(str): Provide the directory consist of images (SAMPLE: ./ROOT_DATA_FOLDER/PRE_FOLDER)
        '''
        
        try:

            for path in os.listdir(IMAGE_FOLDER_PATH):
                img,_= open(path,no_channels,transpose=False)
                pixels = img.reshape(img.shape[0], -1).T 

                mean = np.zeros(pixels.shape[1])
                M2 = np.zeros(pixels.shape[1])

                n = pixels.shape[0]
                new_mean = np.mean(pixels, axis=0)
                new_M2 = np.var(pixels, axis=0) * n

                delta = new_mean - mean
                total_n = n_pixels + n
                mean += delta * n / total_n
                M2 += new_M2 + delta**2 * n_pixels * n / total_n
                n_pixels = total_n

            std = np.sqrt(M2 / n_pixels)
            print(f'[MEAN]----{mean}')
            print(f'[STD]----{std}')
        
        except ValueError:
            print('PATH is invalid !')

In [30]:
process = GEO_PreProssessing()

In [31]:
folderpath =r'C:\Users\b1114674\Desktop\thesis\data\AI4EO'

In [32]:
prelist = [
    os.path.join(folderpath, 'pre', img)
    for img in os.listdir(os.path.join(folderpath, 'pre'))
    if img.endswith('.tif')
]

postlist = [
    os.path.join(folderpath, 'post', img)
    for img in os.listdir(os.path.join(folderpath, 'post'))
    if img.endswith('.tif')
]

In [33]:
for i in range(0,len(prelist)):
    process.makeChips(prelist[i],postlist[i],r'C:\Users\b1114674\Desktop\thesis\data\AI4EO\chippre', r'C:\Users\b1114674\Desktop\thesis\data\AI4EO\chippost')



[OVERLAP ERROR]----- MISMATCH IN IMAGE SIZES
Before PRE crop (6515, 4945, 4)
Before POST crop (6512, 4930, 4)


c:\Users\b1114674\AppData\Local\miniconda3\envs\play\lib\site-packages\geopandas\array.py:340: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may produce unexpected results.
  self.crs = crs
c:\Users\b1114674\AppData\Local\miniconda3\envs\play\lib\site-packages\geopandas\geoseries.py:244: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may produce unexpected results.
  self.geometry.values.crs = value


IMAGE CROPPED TO A COMMON SIZE


AssertionError: ISSUE WITH THE IMAGE SIZING, TRY TO CUT THE IMAGE INTO A COMMAN SIZE (4, 6515, 4945) and (4, 6512, 4930)

In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, calculate_default_transform, Resampling
from shapely.geometry import box, mapping
import os
import numpy as np

class GEO_PreProcessing:

    def __init__(self):
        pass

    def open(self, path, no_channels=[1,2,3], transpose=True):
        """Open an image and optionally select channels"""
        src = rasterio.open(path)
        img = src.read(no_channels)
        if transpose:
            img = np.transpose(img, (1, 2, 0))
        return src, img

    def align_to_ref(self, ref_path, src_path, output_path):
        """Align src_path raster to ref_path raster grid."""
        with rasterio.open(ref_path) as ref, rasterio.open(src_path) as src:
            transform, width, height = calculate_default_transform(
                src.crs, ref.crs, ref.width, ref.height, *ref.bounds
            )
            profile = ref.profile.copy()
            profile.update({
                'transform': ref.transform,
                'crs': ref.crs,
                'width': ref.width,
                'height': ref.height
            })

            data = np.empty((src.count, ref.height, ref.width), dtype=src.dtypes[0])

            for i in range(src.count):
                reproject(
                    source=rasterio.band(src, i + 1),
                    destination=data[i],
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=ref.transform,
                    dst_crs=ref.crs,
                    resampling=Resampling.bilinear
                )

            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(data)

        return output_path

    def makeChips(self, in_pre, in_post, out_folder):
        """Clip both rasters to their common area, ensure same grid, and chip."""
        os.makedirs(out_folder, exist_ok=True)

        # Open the rasters
        pre_src = rasterio.open(in_pre)
        post_src = rasterio.open(in_post)

        # Find intersection (common spatial extent)
        pre_bounds = box(*pre_src.bounds)
        post_bounds = box(*post_src.bounds)
        common_area = pre_bounds.intersection(post_bounds)

        if common_area.is_empty:
            print("[SKIP] No overlap found between pre and post.")
            return

        # Align post to pre grid (auto fix for mismatch)
        aligned_post = os.path.join(out_folder, "temp_aligned_post.tif")
        self.align_to_ref(in_pre, in_post, aligned_post)
        post_src = rasterio.open(aligned_post)

        # Clip both rasters to the same geometry
        pre_clip, _ = mask(pre_src, [mapping(common_area)], crop=True)
        post_clip, _ = mask(post_src, [mapping(common_area)], crop=True)

        # Ensure equal size
        min_h = min(pre_clip.shape[1], post_clip.shape[1])
        min_w = min(pre_clip.shape[2], post_clip.shape[2])
        pre_clip = pre_clip[:, :min_h, :min_w]
        post_clip = post_clip[:, :min_h, :min_w]

        assert pre_clip.shape == post_clip.shape, "Clipped rasters still mismatched after alignment."

        print("[SUCCESS] Images aligned and cropped to same size:", pre_clip.shape)

        # Save or chip further as you already do in your pipeline
        # Example save:
        pre_out = os.path.join(out_folder, "pre_common.tif")
        post_out = os.path.join(out_folder, "post_common.tif")

        with rasterio.open(pre_out, 'w', **pre_src.profile) as dst:
            dst.write(pre_clip)
        with rasterio.open(post_out, 'w', **post_src.profile) as dst:
            dst.write(post_clip)

        print("[DONE] Saved cropped rasters:", pre_out, post_out)
